In [ ]:
import os
from collections import defaultdict
import torch
from tqdm import tqdm
import wandb
import numpy as np

from diffusion_co_design.common import OUTPUT_DIR, get_latest_model, cuda
from diffusion_co_design.rware.schema import (
    ScenarioConfig as RwareScenarioConfig,
    TrainingConfig as RwareTrainingConfig,
    Diffusion as DicodeDesignerConfig,
)
from diffusion_co_design.rware.model.classifier import make_model
from diffusion_co_design.rware.design import DicodeDesigner
from diffusion_co_design.rware.diffusion.generator import Generator, OptimizerDetails
from diffusion_co_design.rware.diffusion.transform import image_projection_constraint

In [ ]:
# D-RWARE Corners Plot
project_name = "diffusion-co-design-rware-rware_16_50_5_4_corners"
api = wandb.Api()
runs = api.runs(path=project_name)

total_steps = 4000
runs_dict = defaultdict(list)
train_reward_key = "train/reward/episode_reward_mean"

for run in tqdm(runs):
    name = run.name
    cfg = run.config

    runs_dict[name].append({"cfg": cfg})
    run_date = run.created_at
    for artifact in run.logged_artifacts():
        if artifact.name.startswith("designer_final"):
            path = f"artifacts/{name}/{run_date}"
            runs_dict[name][-1]["designer_artifact_path"] = path
            if os.path.exists(path):
                continue
            assert artifact.download(root=f"artifacts/{name}/{run_date}") == path

In [ ]:
# Compute proportion of invalid environments for unguided diffusion, low omega, and high omega
scenario_path = "../experiments/train_rware_diffusion/conf/rware_16_50_5_4_corners.yaml"
scenario = RwareScenarioConfig.from_file(
    path=scenario_path, overrides=dict(max_steps=100)
)


representation = "image"
diffusion_dir = os.path.join(
    OUTPUT_DIR, "rware", "diffusion", representation, scenario.name
)


def check_image_validity(image):
    # [C, H, W]
    # No overlaps
    if (image.sum(axis=0) > 1).any():
        return False

    # Correct numbers for each shelf
    shelf_counts = image.sum(axis=(1, 2))

    if tuple(shelf_counts) != (13, 13, 12, 12):
        return False

    # No shelves in corners
    if (
        image[:, 0, 0].any()
        or image[:, 0, -1].any()
        or image[:, -1, 0].any()
        or image[:, -1, -1].any()
    ):
        return False

    return True

In [ ]:
generator = Generator(
    generator_model_path=get_latest_model(diffusion_dir, "model"),
    scenario=scenario,
    representation=representation,
    device=cuda,
)

run_cfg = runs_dict["corners_agent_distill_image"][1]
train_cfg = RwareTrainingConfig.from_raw(run_cfg["cfg"])

model = make_model(
    model=train_cfg.designer.model.name,
    scenario=scenario,
    model_kwargs=train_cfg.designer.model.model_kwargs,
    device=cuda,
)
state_dict = torch.load(
    os.path.join(run_cfg["designer_artifact_path"], "designer_3999.pt"),
    map_location=cuda,
)
model.load_state_dict(state_dict)
model = model.eval()

# Low weight guided batch
for weight in [0, 10, 50, 100, 500]:
    batch = []
    for i in range(4):
        designer_cfg = train_cfg.designer
        assert isinstance(designer_cfg, DicodeDesignerConfig)

        operation = OptimizerDetails()
        operation.lr = 0
        operation.backward_steps = 0
        operation.num_recurrences = 1
        operation.forward_guidance_wt = weight
        operation.projection_constraint = None

        minibatch = generator.generate_batch(
            batch_size=32, value=model, use_operation=True, operation_override=operation
        )
        minibatch = minibatch.round()
        batch.append(minibatch)
    batch = np.concatenate(batch, axis=0)

    valid_percentage = sum(check_image_validity(img) for img in batch) / len(batch)
    print(f"{weight} valid percentage: {valid_percentage:.2%}")

# Recurrent steps 8
# 0 valid percentage: 98.44%
# 10 valid percentage: 98.44%
# 50 valid percentage: 98.44%
# 100 valid percentage: 96.88%
# 500 valid percentage: 87.50%

# Recurrence steps 1
# 0 valid percentage: 72.66%
# 10 valid percentage: 72.66%
# 50 valid percentage: 60.94%
# 100 valid percentage: 23.44%
# 500 valid percentage: 0.00%

In [ ]:
from diffusion_co_design.wfcrl.design import (
    RandomDesigner as WfcrlRandomDesigner,
    DicodeDesigner as WfcrlDicodeDesigner,
    SamplingDesigner as WfcrlSamplingDesigner,
    DescentDesigner as WfcrlDescentDesigner,
)

from diffusion_co_design.wfcrl.diffusion.generator import eval_to_train, train_to_eval
from diffusion_co_design.wfcrl.model.rl import maybe_make_denormaliser
from diffusion_co_design.wfcrl.model.classifier import GNNCritic
from diffusion_co_design.wfcrl.schema import (
    TrainingConfig as WfcrlTrainingConfig,
    Diffusion as WfcrlDiffusionDesignerConfig,
)
from diffusion_co_design.common.design import DesignerParams
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
# PUG Ablation (WFCRL)
api = wandb.Api()
runs = api.runs(path="diffusion-co-design-wfcrl-wfcrl_8")
total_steps = 301
wfcrl_runs = defaultdict(list)
for run in tqdm(runs):
    name = run.name
    cfg = run.config

    run_data = {"cfg": cfg}
    run_data["designer_state_dict"] = None
    for artifact in run.logged_artifacts():
        if artifact.name.startswith("designer_final"):
            artifact_dir = artifact.download()
            state_dict = torch.load(
                os.path.join(artifact_dir, "designer_300.pt"), map_location=cuda
            )
            run_data["designer_state_dict"] = state_dict

    wfcrl_runs[name].append(run_data)


In [ ]:
wfcrl_run_config = wfcrl_runs["wfcrl_diffusion_distill"][0]
wfcrl_train_config = WfcrlTrainingConfig.from_raw(wfcrl_run_config["cfg"])
state_dict = wfcrl_run_config["designer_state_dict"]

In [ ]:
N = 64

scenario = wfcrl_train_config.scenario
designer_cfg = wfcrl_train_config.designer
assert isinstance(designer_cfg, WfcrlDiffusionDesignerConfig)
designer_setting = DesignerParams.placeholder(scenario=scenario)

critic = GNNCritic(
    cfg=scenario,
    node_emb_dim=designer_cfg.model.node_emb_size,
    n_layers=designer_cfg.model.depth,
    post_hook=maybe_make_denormaliser(normalisation=wfcrl_train_config.normalisation),
).to(device=cuda)

critic.load_state_dict(state_dict)

random_designer = WfcrlRandomDesigner(designer_setting=designer_setting, seed=0)
random_batch = random_designer.generate_random_layouts(batch_size=64)
random_batch = eval_to_train(torch.stack(random_batch).to(device=cuda), cfg=scenario)
random_values = critic(random_batch).detach().cpu().numpy()
print(random_values.mean())


In [ ]:
dicode_designer = WfcrlDicodeDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    diffusion=designer_cfg.diffusion,
    normalisation_statistics=wfcrl_train_config.normalisation,
    device=cuda,
)
dicode_designer.diffusion.forward_guidance_annealing = False
dicode_designer.model.load_state_dict(state_dict)
dicode_batch = dicode_designer._generate_layout_batch(batch_size=64)
dicode_batch = eval_to_train(
    torch.tensor(np.array(dicode_batch)).to(device=cuda), cfg=scenario
)
dicode_values = critic(dicode_batch).detach().cpu().numpy()
print(dicode_values.mean())

In [ ]:
ug_designer = WfcrlDicodeDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    diffusion=designer_cfg.diffusion,
    normalisation_statistics=wfcrl_train_config.normalisation,
    device=cuda,
)


def pc(x):
    return x


ug_designer.pc = pc
ug_designer.diffusion.forward_guidance_annealing = False
ug_designer.model.load_state_dict(state_dict)
dicode_designer.projection_constraint = pc
dicode_designer.pc = pc
ug_batch = dicode_designer._generate_layout_batch(batch_size=64)
ug_batch = eval_to_train(torch.tensor(np.array(ug_batch)).to(device=cuda), cfg=scenario)
ug_values = critic(ug_batch).detach().cpu().numpy()
print(ug_values.mean())

In [ ]:
sampling_designer = WfcrlSamplingDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    normalisation_statistics=wfcrl_train_config.normalisation,
    n_samples=32,
    device=cuda,
)
sampling_designer.model.load_state_dict(state_dict)
sampling_batch = sampling_designer._generate_layout_batch(batch_size=64)
sampling_batch = eval_to_train(torch.stack(sampling_batch), cfg=scenario).to(
    device="cuda"
)
sampling_values = critic(sampling_batch).detach().cpu().numpy()
print(sampling_values.mean())


In [ ]:
descent_designer = WfcrlDescentDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    normalisation_statistics=wfcrl_train_config.normalisation,
    device=cuda,
    n_gradient_iterations=20,
)

descent_designer.model.load_state_dict(state_dict)
descent_batch = descent_designer._generate_layout_batch(batch_size=64)
descent_batch = eval_to_train(torch.stack(descent_batch), cfg=scenario)
descent_values = critic(descent_batch).detach().cpu().numpy()
print(descent_values.mean())

In [ ]:
sns.set_theme(style="whitegrid")

labels = []
exp_returns = []


selected_envs = {}

for label, y in (
    ("UG", ug_values),
    ("PUG", dicode_values),
    ("Descent", descent_values),
    ("Sampling", sampling_values),
    ("DR", random_values),
):
    labels.append(label)
    exp_returns.append(y)
    selected_envs[label] = {"best_idx": y.argmax(), "worst_idx": y.argmin()}

    mean = y.mean()
    std = y.std(ddof=1)  # sample std
    se = std / np.sqrt(len(y))  # standard error
    ci95 = 1.96 * se  # 95% confidence interval

    print(f"{label}: mean={mean:.4f}, std={std:.4f}, se={se:.4f}, ci95={ci95:.4f}")

colors = sns.color_palette(n_colors=len(exp_returns))

fig, ax = plt.subplots(figsize=(5, 2.6))
box = ax.boxplot(
    exp_returns,
    patch_artist=True,
    labels=labels,
    boxprops=dict(linewidth=1.2),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="gray"),
    capprops=dict(color="gray"),
)

# Apply colors
for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)


ax.set_title("Environment Search Comparison")
ax.set_ylabel("Critic Value")
ax.set_xlabel("Generator Method")
fig.tight_layout()
fig.savefig(fname="ablation-pug-wfcrl-box.png", bbox_inches="tight", dpi=300)

# Designer Ablation (VMAS)

In [ ]:
import os
from collections import defaultdict
import torch
from tqdm import tqdm
import wandb
import numpy as np
from diffusion_co_design.vmas.design import (
    RandomDesigner as VmasRandomDesigner,
    DicodeDesigner as VmasDicodeDesigner,
    SamplingDesigner as VmasSamplingDesigner,
    DescentDesigner as VmasDescentDesigner,
)
from diffusion_co_design.vmas.diffusion.generator import (
    eval_to_train as vmas_eval_to_train,
)
from diffusion_co_design.vmas.model.classifier import GNNEnvCritic, MLPEnvCritic
from diffusion_co_design.vmas.schema import (
    TrainingConfig as VmasTrainingConfig,
    GlobalPlacementScenarioConfig,
    LocalPlacementScenarioConfig,
    SimpleSpreadScenarioConfig,
)
from diffusion_co_design.common.design import DesignerParams

In [ ]:
# Load VMAS runs
api = wandb.Api()
vmas_project_name = "diffusion-co-design-vmas-obstacle_navigation_3"
runs = api.runs(path=vmas_project_name)

total_steps = 201
vmas_runs = defaultdict(list)

for run in tqdm(runs):
    name = run.name
    cfg = run.config

    run_data = {"cfg": cfg}
    run_data["designer_state_dict"] = None
    for artifact in run.logged_artifacts():
        if artifact.name.startswith("designer_final"):
            artifact_dir = artifact.download()
            state_dict = torch.load(
                os.path.join(artifact_dir, f"designer_{total_steps - 1}.pt"),
                map_location=cuda,
            )
            run_data["designer_state_dict"] = state_dict

    vmas_runs[name].append(run_data)

In [ ]:
vmas_run_config = vmas_runs["vmas_diffusion_duplicate"][0]
vmas_train_config = VmasTrainingConfig.from_raw(vmas_run_config["cfg"])
state_dict = vmas_run_config["designer_state_dict"]

In [ ]:
N = 64

scenario = vmas_train_config.scenario
designer_cfg = vmas_train_config.designer
designer_setting = DesignerParams.placeholder(scenario=scenario)

# Build a standalone value model (same architecture as the one inside the designer)
if isinstance(scenario, GlobalPlacementScenarioConfig):
    critic = GNNEnvCritic(
        scenario=scenario,
        node_emb_dim=designer_cfg.model.hidden_size,
        num_layers=designer_cfg.model.depth,
        k=designer_cfg.model.k,
    ).to(device=cuda)
else:
    critic = MLPEnvCritic(
        scenario=scenario,
        hidden_dim=designer_cfg.model.hidden_size,
        num_layers=designer_cfg.model.depth,
    ).to(device=cuda)

critic.load_state_dict(state_dict)
critic = critic.eval()

# Random baseline
random_designer = VmasRandomDesigner(designer_setting=designer_setting, seed=0)
random_layouts = random_designer.generate_random_layouts(batch_size=N)
random_batch = vmas_eval_to_train(
    torch.stack(random_layouts).to(device=cuda), cfg=scenario
)
with torch.no_grad():
    random_values = (
        critic.predict_theta_value(random_batch).squeeze().detach().cpu().numpy()
    )
print(f"Random mean: {random_values.mean():.4f}")

In [ ]:
from diffusion_co_design.vmas.schema import Diffusion as VmasDiffusionConfig

assert isinstance(designer_cfg, VmasDiffusionConfig)

diffusion = designer_cfg.diffusion

dicode_designer = VmasDicodeDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    diffusion=designer_cfg.diffusion,
    device=cuda,
)
dicode_designer.diffusion.forward_guidance_annealing = False
dicode_designer.model.load_state_dict(state_dict)
dicode_batch = dicode_designer._generate_layout_batch(batch_size=N)
dicode_batch = vmas_eval_to_train(
    torch.stack(dicode_batch).to(device=cuda), cfg=scenario
)
with torch.no_grad():
    dicode_values = (
        critic.predict_theta_value(dicode_batch).squeeze().detach().cpu().numpy()
    )
print(f"DiCoDe mean: {dicode_values.mean():.4f}")

In [ ]:
sampling_designer = VmasSamplingDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    n_samples=32,
    device=cuda,
)
sampling_designer.model.load_state_dict(state_dict)
sampling_batch = sampling_designer._generate_layout_batch(batch_size=N)
sampling_batch = vmas_eval_to_train(
    torch.stack(sampling_batch).to(device=cuda), cfg=scenario
)
with torch.no_grad():
    sampling_values = (
        critic.predict_theta_value(sampling_batch).squeeze().detach().cpu().numpy()
    )
print(f"Sampling mean: {sampling_values.mean():.4f}")

In [ ]:
descent_designer = VmasDescentDesigner(
    designer_setting=designer_setting,
    classifier=designer_cfg.model,
    n_gradient_iterations=20,
    lr=0.1,
    device=cuda,
)
descent_designer.model.load_state_dict(state_dict)
descent_batch = descent_designer._generate_layout_batch(batch_size=N)
descent_batch = vmas_eval_to_train(
    torch.stack(descent_batch).to(device=cuda), cfg=scenario
)
with torch.no_grad():
    descent_values = (
        critic.predict_theta_value(descent_batch).squeeze().detach().cpu().numpy()
    )
print(f"Descent mean: {descent_values.mean():.4f}")

In [ ]:
sns.set_theme(style="whitegrid")

labels = []
exp_returns = []

for label, y in (
    ("DiCoDe", dicode_values),
    ("Descent", descent_values),
    ("Sampling", sampling_values),
    ("DR", random_values),
):
    labels.append(label)
    exp_returns.append(y)

    mean = y.mean()
    std = y.std(ddof=1)
    se = std / np.sqrt(len(y))
    ci95 = 1.96 * se
    print(f"{label}: mean={mean:.4f}, std={std:.4f}, se={se:.4f}, ci95={ci95:.4f}")

colors = sns.color_palette(n_colors=len(exp_returns))

fig, ax = plt.subplots(figsize=(5, 2.6))
box = ax.boxplot(
    exp_returns,
    patch_artist=True,
    labels=labels,
    boxprops=dict(linewidth=1.2),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="gray"),
    capprops=dict(color="gray"),
)

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)

ax.set_title("Environment Search Comparison (VMAS)")
ax.set_ylabel("Critic Value")
ax.set_xlabel("Generator Method")
fig.tight_layout()
fig.savefig(fname="ablation-designers-vmas-box.png", bbox_inches="tight", dpi=300)